# 29D — Creator benchmark review console

**Runtime: CPU. Run all once.** Upload the `calorie_creator_label_review_queue.zip` produced by Notebook 29C. The notebook downloads only archived RGB/pose evidence; it does not download YouTube video, extract pose, train, score, inspect predictions, or lock the benchmark.

`confirmation` carries forward our existing first-pass decisions as Reviewer A. A distinct reviewer can choose `blind`, which hides those decisions. Neither output is `benchmark_ready` until adjudication.

In [ ]:
REVIEW_MODE='confirmation'  # confirmation now; blind only for a distinct reviewer
REVIEWER_ID='reviewer_a'

import hashlib,json,math,shutil,urllib.request,zipfile
from pathlib import Path
import pandas as pd
import ipywidgets as w
from IPython.display import Image,clear_output,display
from google.colab import files
assert REVIEW_MODE in {'confirmation','blind'}
SOURCE_COMMIT='a2b33753a43835fee1e45191151c6958a1ec412f'
RAW=f'https://raw.githubusercontent.com/bariscelikk1/cALorie/{SOURCE_COMMIT}'
WORK=Path('/content/calorie_29d_review')
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir()

def sha(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''): h.update(b)
 return h.hexdigest()
def extract(zpath,dest):
 dest=Path(dest).resolve();dest.mkdir(parents=True,exist_ok=True)
 with zipfile.ZipFile(zpath) as z:
  for m in z.infolist():
   target=(dest/m.filename).resolve()
   if target!=dest and dest not in target.parents: raise ValueError('Unsafe ZIP path')
  z.extractall(dest)
def verify(root):
 manifest=Path(root)/'SHA256SUMS.txt'; checked=0
 if not manifest.exists(): return checked
 for line in manifest.read_text().splitlines():
  if not line.strip(): continue
  expected,rel=line.split(maxsplit=1);p=Path(root)/rel.strip()
  if sha(p)!=expected: raise ValueError(f'Checksum mismatch: {rel}')
  checked+=1
 return checked

print('Upload calorie_creator_label_review_queue.zip')
uploaded=files.upload(); names=[n for n in uploaded if n.endswith('.zip')]
assert len(names)==1,'Upload exactly one queue ZIP'
qzip=WORK/'queue.zip';qzip.write_bytes(uploaded[names[0]])
extract(qzip,WORK/'queue');assert verify(WORK/'queue')>=10
sources={
 'dense':'ml/artifacts/youtube_creator_benchmark_v1/dense_review_evidence/calorie_creator_benchmark_dense_review_evidence.zip',
 'supplemental':'ml/artifacts/youtube_creator_benchmark_v1/supplemental_pose_review.zip'}
for name,rel in sources.items():
 target=WORK/f'{name}.zip';print('Downloading archived evidence:',name)
 urllib.request.urlretrieve(f'{RAW}/{rel}',target);extract(target,WORK/name)
 print(name,'verified files:',verify(WORK/name))
queue=pd.read_csv(WORK/'queue'/'draft_review_queue.csv')
classes=json.loads((WORK/'queue'/'classes.json').read_text())['exercises']
assert len(queue)==41 and queue.video_id.nunique()==6 and len(classes)==14
assert queue.review_status.eq('draft').all() and (~queue.benchmark_ready).all()
print(f'Ready: 41 regions / 6 videos / mode={REVIEW_MODE}')

## Review

Use only visible motion. `accept` is one clear supported exercise; `unknown` covers rest/setup/transition/ambiguous or unsupported motion; `relabel` changes to another supported class; `split` needs multiple segments; `reject` is unusable. Download progress before closing because rerunning clears in-memory answers.

In [ ]:
draft_to_decision={'trim_candidate':'accept','relabel_unknown':'unknown','reject_candidate':'reject','variant_review':'needs_review','requires_split':'split','supplemental_pose_candidate':'needs_review'}
answers={};state={'i':0}
title=w.HTML();prior=w.HTML();source_link=w.HTML();status=w.HTML()
page=w.Dropdown(description='Evidence:',layout=w.Layout(width='90%'));picture=w.Output()
decision=w.Dropdown(options=['','accept','unknown','relabel','split','reject','needs_review'],description='Decision:')
label=w.Dropdown(options=['']+classes+['unknown'],description='Label:')
start=w.FloatText(description='Start s:');end=w.FloatText(description='End s:')
segment=w.Dropdown(options=['','exercise','unknown','transition','compound','unusable'],description='Segment:')
confidence=w.Dropdown(options=['','high','medium','low'],description='Confidence:')
notes=w.Textarea(description='Notes:',layout=w.Layout(width='90%',height='80px'))
back=w.Button(description='← Previous');save=w.Button(description='Save + next',button_style='success');download=w.Button(description='Download progress ZIP',button_style='info')

def show_page(change=None):
 with picture:
  clear_output(wait=True)
  if page.value: display(Image(filename=page.value,width=1100))
page.observe(show_page,names='value')
def evidence(row):
 root=WORK/'dense'/'dense_contact_sheets' if str(row.evidence_path).startswith('dense') else WORK/'supplemental'/'pose_contact_sheets'
 return sorted((root/row.interval_id).glob('*.jpg'))
def defaults(row):
 if REVIEW_MODE=='blind': return {'decision':'','label':'','start':float(row.locator_start_s),'end':float(row.locator_end_s),'segment':'','confidence':'','notes':''}
 d=draft_to_decision[row.first_pass_decision];lab='unknown' if d in {'unknown','reject'} else row.provisional_class
 s=row.suggested_core_start_s if pd.notna(row.suggested_core_start_s) else row.locator_start_s
 e=row.suggested_core_end_s if pd.notna(row.suggested_core_end_s) else row.locator_end_s
 return {'decision':d,'label':lab,'start':float(s),'end':float(e),'segment':'exercise' if d=='accept' else ('unknown' if d in {'unknown','reject'} else ''),'confidence':'low' if row.first_pass_decision=='supplemental_pose_candidate' else 'medium','notes':''}
def render(i):
 state['i']=max(0,min(i,len(queue)-1));row=queue.iloc[state['i']];a=answers.get(row.interval_id,defaults(row))
 title.value=f'<h3>{state["i"]+1}/41 — {row.interval_id}</h3><b>{row.creator}</b> · locator {row.locator_start_s:.0f}–{row.locator_end_s:.0f}s'
 prior.value=(f'<b>Existing draft:</b> {row.original_proposal} / {row.first_pass_decision}<br><b>Earlier note:</b> {row.notes}' if REVIEW_MODE=='confirmation' else '<i>Blind mode: prior label and decision hidden.</i>')
 source_link.value=f'<a href="https://www.youtube.com/watch?v={row.video_id}&t={int(row.locator_start_s)}s" target="_blank">Optional source timestamp</a>'
 decision.value=a['decision'];label.value=a['label'];start.value=a['start'];end.value=a['end'];segment.value=a['segment'];confidence.value=a['confidence'];notes.value=a['notes']
 pages=evidence(row);page.options=[(p.name,str(p)) for p in pages]
 status.value=f'Saved {len(answers)}/41 · evidence pages here: {len(pages)}' if pages else '<b style="color:red">Evidence missing.</b>'
 if pages: page.value=str(pages[0]);show_page()
def frame():
 out=queue.copy();out['reviewer_id']=REVIEWER_ID;out['review_mode']=REVIEW_MODE
 for c in ['reviewed_motion_label','reviewed_start_s','reviewed_end_s','segment_type','confidence','decision','review_notes']:out[c]=''
 out['review_completed']=False;out['needs_second_review']=True;out['benchmark_ready']=False
 for i,row in out.iterrows():
  if row.interval_id not in answers:continue
  a=answers[row.interval_id];out.at[i,'reviewed_motion_label']=a['label'];out.at[i,'reviewed_start_s']=a['start'];out.at[i,'reviewed_end_s']=a['end'];out.at[i,'segment_type']=a['segment'];out.at[i,'confidence']=a['confidence'];out.at[i,'decision']=a['decision'];out.at[i,'review_notes']=a['notes'];out.at[i,'review_completed']=True
 return out
def export(do_download=False):
 outdir=WORK/'output';outdir.mkdir(exist_ok=True);f=frame();f.to_csv(outdir/'review_progress.csv',index=False)
 meta={'notebook':'29d_colab_creator_benchmark_review.ipynb','source_commit':SOURCE_COMMIT,'review_mode':REVIEW_MODE,'reviewer_id':REVIEWER_ID,'draft_regions':41,'completed_rows':int(f.review_completed.sum()),'benchmark_ready':False,'model_predictions_reviewed':False,'labels_locked':False}
 (outdir/'run_metadata.json').write_text(json.dumps(meta,indent=2)+'\n');(outdir/'README.md').write_text('Reviewer progress only. This output does not lock the benchmark or permit model selection.\n')
 sums=[f'{sha(p)}  {p.name}' for p in sorted(outdir.glob('*')) if p.name!='SHA256SUMS.txt'];(outdir/'SHA256SUMS.txt').write_text('\n'.join(sums)+'\n')
 archive=Path(shutil.make_archive('/content/calorie_creator_review_progress','zip',outdir))
 if do_download:files.download(str(archive))
 return archive
def save_current(_=None):
 row=queue.iloc[state['i']]
 if not REVIEWER_ID.strip() or not decision.value or not confidence.value:status.value='<b style="color:red">Reviewer, decision and confidence are required.</b>';return
 chosen=label.value
 if decision.value in {'accept','relabel'}:
  if chosen not in classes:status.value='<b style="color:red">Choose a supported label.</b>';return
  if not(start.value<end.value and start.value>=row.locator_start_s and end.value<=row.locator_end_s):status.value='<b style="color:red">Accepted boundaries must stay inside locator.</b>';return
 if decision.value in {'unknown','reject'}:chosen='unknown'
 answers[row.interval_id]={'decision':decision.value,'label':chosen,'start':float(start.value),'end':float(end.value),'segment':segment.value,'confidence':confidence.value,'notes':notes.value.strip()}
 export(False);render(state['i']+1);status.value=f'<b>Saved {len(answers)}/41.</b>'
back.on_click(lambda _:render(state['i']-1));save.on_click(save_current);download.on_click(lambda _:export(True))
display(title,prior,source_link,page,picture,w.HBox([decision,label]),w.HBox([start,end]),w.HBox([segment,confidence]),notes,w.HBox([back,save,download]),status);render(0)